In [5]:
import duckdb
import pandas as pd

# connect to an in-memory database
con = duckdb.connect()

# create a SQL view of the parquet files
con.execute("""
    CREATE OR REPLACE VIEW employees AS 
    SELECT * FROM read_parquet('../data/processed/*.parquet')
""")

print("Database View Created! You can now run SQL.")

Database View Created! You can now run SQL.


In [6]:
# run a simple SQL query
con.sql("SELECT * FROM employees LIMIT 5").show()

┌──────────────────────────────────────┬────────────┬───────────┬─────────────────────────┬─────────────────────┬─────────────┬──────────┬─────────────────┬──────────────┐
│             employee_id              │ first_name │ last_name │          email          │      hire_date      │ department  │  salary  │ office_location │ tenure_years │
│               varchar                │  varchar   │  varchar  │         varchar         │    timestamp_ns     │   varchar   │  double  │     varchar     │    double    │
├──────────────────────────────────────┼────────────┼───────────┼─────────────────────────┼─────────────────────┼─────────────┼──────────┼─────────────────┼──────────────┤
│ 59a37d75-cf73-43f5-af01-d989d8698973 │ Bryan      │ Kline     │ smithjoel@example.net   │ 2023-10-07 00:00:00 │ Sales       │ 116017.0 │ London          │          2.3 │
│ 3577cdb1-afb1-4ab5-955d-ad83e2ce658c │ Gloria     │ Goodwin   │ curryeric@example.org   │ 2021-06-09 00:00:00 │ Marketing   │ 58098.45 │ L

### Example ad hoc queries from stakeholders

In [7]:
'''
Business Question 1 - Salary by Department
'''

# write the query
query = """
    SELECT 
        department,
        COUNT(*) as employee_count,
        ROUND(AVG(salary), 2) as avg_salary,
        MIN(salary) as min_salary,
        MAX(salary) as max_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
"""

# turn the result into a dataframe
con.sql(query).df()

,department,employee_count,avg_salary,min_salary,max_salary
0,Product,105,104823.64,51112.99,148182.70
1,Engineering,103,102814.05,50300.88,149565.12
2,Sales,90,102757.63,51319.54,149117.13
3,HR,103,100952.59,50456.84,149216.86
4,Marketing,99,98284.23,50033.70,149375.98


In [10]:
'''
Business Question 2 - Tenure Analysis
'''
# write the query
query = """
    SELECT 
        first_name, 
        last_name, 
        department, 
        tenure_years,
        office_location
    FROM employees
    WHERE tenure_years  > 3
    ORDER BY tenure_years DESC
    LIMIT 10
"""

# turn the result into a dataframe
con.sql(query).df()

,first_name,last_name,department,tenure_years,office_location
0,Brittany,Clark,Sales,5.0,Remote
1,Lindsay,Gibson,Engineering,5.0,Remote
2,Jeremy,Wilkins,Sales,5.0,Remote
3,Paul,Mitchell,Product,5.0,Remote
4,Jeffrey,Jones,Marketing,4.9,New York
5,Brian,English,Product,4.9,New York
6,Rebecca,Carter,Product,4.9,Remote
7,Scott,Young,Marketing,4.9,London
8,Crystal,Campbell,HR,4.9,Remote
9,Michael,Fisher,HR,4.9,New York
